# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [7]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import AsyncOpenAI

In [8]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if api_key and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = "gemini-3.1-flash-lite"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = AsyncOpenAI(api_key=api_key, base_url=GEMINI_BASE_URL)

API key looks good so far


In [8]:
links = fetch_website_links("https://www.vincentinc.com/")
links

['/cart',
 '#page',
 '/',
 '/our-story',
 '/contact-us',
 '/testimonials',
 '/main-fr',
 '/',
 '/our-story',
 '/contact-us',
 '/testimonials',
 '/main-fr',
 '/our-story',
 '/contact-us',
 '/testimonials',
 '/main-fr',
 '#section1',
 '/it-consulting-support',
 '/it-consulting-support',
 '/it-consulting-support',
 'data-management',
 '/data-management',
 '/solutions-development',
 '/solutions-development',
 '/customer-relationship-management',
 '/customer-relationship-management',
 '/communications-strategy',
 '/communications-strategy',
 '#sectionF',
 '#menu01',
 '#menu02',
 '#menu03',
 '#menu04',
 '#menu05',
 '#menu01',
 '#menu02',
 '#menu03',
 '#menu04',
 '#menu05',
 '/our-story',
 '/best-practices-security-and-privacy',
 '/privacy-policy',
 '/careers',
 '/s/AccessibilityPlan.pdf',
 '/s/CustomerServicePlan.pdf',
 '/it-consulting-support',
 '/data-management',
 '/solutions-development',
 '/customer-relationship-management',
 '/communications-strategy',
 'tel:4164455443',
 'tel:88832891

## First step: Have ~~GPT-5-nano~~ Gemini Flash figure out which links are relevant

### Use a call to ~~gpt-5-nano~~ gemini-3.1-flash-lite to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [9]:
link_system_prompt = """
    You are provided with a list of links found on a webpage.
    You are able to decide which of the links would be most relevant to include in a brochure about the company,
    such as links to the company's story and the Customer Relationship Management solution.
    You should respond in JSON as in this example:

    {
        "links": [
            {"type": "home", "url": "https://www.vincentinc.com"},
            {"type": "our story", "url": "https://www.vincentinc.com/our-story"},
            {"type": "CRM", "url": "https://www.vincentinc.com/customer-relationship-management"}
        ]
    }
"""

In [14]:
def get_links_user_prompt(url):
    user_prompt = f"""
        Here is the list of links on the website {url} -
        Please decide which of these are relevant web links for a brochure about the company, 
        respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):

    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [11]:
print(get_links_user_prompt("https://www.vincentinc.com"))


        Here is the list of links on the website https://www.vincentinc.com -
        Please decide which of these are relevant web links for a brochure about the company, 
        respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):

    /cart
#page
/
/our-story
/contact-us
/testimonials
/main-fr
/
/our-story
/contact-us
/testimonials
/main-fr
/our-story
/contact-us
/testimonials
/main-fr
#section1
/it-consulting-support
/it-consulting-support
/it-consulting-support
data-management
/data-management
/solutions-development
/solutions-development
/customer-relationship-management
/customer-relationship-management
/communications-strategy
/communications-strategy
#sectionF
#menu01
#menu02
#menu03
#menu04
#menu05
#menu01
#menu02
#menu03
#menu04
#menu05
/our-story
/best-practices-security-and-privacy
/privacy-policy
/careers
/s/AccessibilityPlan.pdf
/s/CustomerServicePlan.pdf
/it-consul

In [12]:
async def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = await gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links
    

In [13]:
await select_relevant_links("https://www.vincentinc.com")

{'links': [{'type': 'home', 'url': 'https://www.vincentinc.com/'},
  {'type': 'our story', 'url': 'https://www.vincentinc.com/our-story'},
  {'type': 'contact us', 'url': 'https://www.vincentinc.com/contact-us'},
  {'type': 'testimonials', 'url': 'https://www.vincentinc.com/testimonials'},
  {'type': 'IT consulting support',
   'url': 'https://www.vincentinc.com/it-consulting-support'},
  {'type': 'data management',
   'url': 'https://www.vincentinc.com/data-management'},
  {'type': 'solutions development',
   'url': 'https://www.vincentinc.com/solutions-development'},
  {'type': 'customer relationship management',
   'url': 'https://www.vincentinc.com/customer-relationship-management'},
  {'type': 'communications strategy',
   'url': 'https://www.vincentinc.com/communications-strategy'},
  {'type': 'best practices security',
   'url': 'https://www.vincentinc.com/best-practices-security-and-privacy'},
  {'type': 'careers', 'url': 'https://www.vincentinc.com/careers'}]}

In [15]:
await select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 9 relevant links


{'links': [{'type': 'home', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'about us', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'inference endpoints', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'careers', 'url': 'https://apply.workable.com/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to ~~GPT-5-nano~~ Gemini Flash

In [10]:
async def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = await select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [27]:
import tracemalloc
tracemalloc.start() # Get more detailed information about where the memory allocation issue is occurring

print(await fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4.1-Flash-1017
Updated
8 days ago
•
73.2k
•
2.74k
RecursiveSI/flint
Updated
3 days ago
•
4.8k
•
753
black-forest-labs/flux-3-image-klein-comfy
Updated
8 days ago
•
591k
•
1.48k
Lightricks/LTX-2.5
Updated
14 days ago
•
2.09M
•
9.15k
OpenMOSS-Team/MOSS-I

In [28]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of the company's story and competitive advantage, focusing on the Customer Relationship Management solution.
"""

In [5]:
async def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
        You are looking at a company called: {company_name}
        Here are the contents of its landing page and other relevant pages;
        use this information to build a short brochure of the company in markdown without code blocks.\n\n
    """
    user_prompt += await fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [30]:
async def create_brochure(company_name, url):
    response = await gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": await get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [31]:
await create_brochure("Vincent Associates", "https://www.vincentinc.com/")

Selecting relevant links for https://www.vincentinc.com/ by calling gemini-3.1-flash-lite
Found 11 relevant links


# Vincent Associates: Your Data Management Professionals

Since 1995, Vincent Associates has empowered organizations to achieve operational excellence through customized, innovative management solutions. With nearly three decades of industry experience, we serve as a trusted partner for businesses ranging from retail and finance to government and non-profit sectors, providing the tools necessary to turn raw data into strategic assets.

## The Vincent Advantage

At Vincent Associates, we combine technical mastery with a deep understanding of organizational dynamics. Our approach is defined by three core pillars:

* **Extensive Experience:** We provide high-quality, bilingual (English/French) solutions. Beyond data analytics and IT consulting, we offer professional translation services to ensure your marketing initiatives and web presence resonate globally.
* **Deep Knowledge:** We don’t just provide software; we partner with you to design and implement systems that secure your information flow. By blending state-of-the-art technology with "outside-the-box" thinking, we bridge the gap between complex data and actionable insights.
* **Proven Results:** Our commitment to high-value, top-shelf consulting ensures that your investment in our systems yields measurable improvements in operational effectiveness.

## Tailored Customer Relationship Management (CRM)

In today’s competitive landscape, your customer data is your most valuable resource. Vincent Associates specializes in building custom CRM applications designed to meet the unique requirements of your business, regardless of your sector.

Our CRM solutions go beyond simple data storage. We integrate your CRM with intelligent communications and marketing strategies, allowing you to:

* **Measure and Improve:** Utilize data-driven metrics to refine your operational effectiveness.
* **Engage Responsively:** Leverage your internal data to build effective, responsive communication strategies that deepen customer loyalty.
* **Stay Flexible:** Benefit from bespoke applications that scale with your organization, ensuring that your technology works for you—not the other way around.

## Join Our Journey
Whether you are looking for a reliable IT consulting partner, a secure data warehousing solution, or a sophisticated CRM platform to transform your customer interactions, Vincent Associates provides the expertise and innovation to help you succeed. 

Partner with us to secure your data, optimize your operations, and drive your business forward.

**Contact Vincent Associates today to discuss your next project.**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from ~~OpenAI~~ Gemini,
with the familiar typewriter animation

In [2]:
link_system_prompt = """
    You are provided with a list of links found on a webpage.
    You are able to decide which of the links would be most relevant to include in a brochure about the company,
    such as links to the company's products and services, news and events, job openings.
    You should respond in JSON as in this example:

    {
        "links": [
            {"type": "about", "url": "https://www.fundata.com/about"},
            {"type": "analytics", "url": "https://www.fundata.com/analytics"},
            {"type": "careers", "url": "https://www.fundata.com/careers"},
        ]
    }
"""

In [3]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of the company's history, resources, focusing on the analytics services.
"""

In [4]:
async def stream_brochure(company_name, url):
    stream = await gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": await get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)

    async for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)

In [15]:
await stream_brochure("Fundata", "https://www.fundata.com/")

Selecting relevant links for https://www.fundata.com/ by calling gemini-3.1-flash-lite
Found 15 relevant links


# Fundata Canada Inc.
### Data That Matters

## Our Story
Founded in 1987, Fundata began with a singular focus: collecting and distributing investment fund pricing to local media and businesses. Over the past three decades, we have evolved from a local provider into a multi-national leader in data distribution and investment fund analytics. Today, our network powers information for over 80,000 advisors and millions of investors, serving as a cornerstone of the North American financial landscape.

## Our Mission
We strive to be more than a vendor; we aim to be your most trusted business partner. Whether providing raw data feeds or fully hosted web solutions, our commitment remains the same: delivering the most timely, accurate, and comprehensive investment information available, backed by a culture of flexibility and reliable client service.

## Analytics Services
At the heart of Fundata’s value proposition is our robust suite of analytical tools, designed to turn complex market data into actionable intelligence:

*   **FundGrade Ratings:** An objective, quantitative measure that helps investors identify superior fund performance within the Canadian investment universe.
*   **ESG Ratings:** Specialized tools designed to integrate Environmental, Social, and Governance criteria into investment analysis.
*   **Fixed Income Analytics:** Precision-driven data tools tailored for the nuances of fixed-income markets.
*   **Performance Analytics:** Comprehensive assessment tools that allow advisors and institutions to evaluate funds with depth and clarity.
*   **Marketing & Web Solutions:** Turn-key solutions, including automated Marketing Sheets and web-based integration, that allow our clients to present professional, data-rich analysis to their own stakeholders effortlessly.

## Resources & Expertise
Fundata serves as a vital bridge in the financial industry. Beyond our proprietary data, we provide a wealth of resources, including:
*   **Industry Connectivity:** We work closely with regulatory bodies and industry associations to maintain the highest standards of data integrity.
*   **Educational Support:** We provide ongoing educational resources to ensure our partners stay ahead of market trends and regulatory shifts.
*   **Market Reach:** Our data feeds and analytical engines are integrated into major media outlets, institutional back offices, and front-end advisor platforms throughout North America.

## Join Us
Fundata is a growing organization committed to innovation and excellence. We are constantly looking for talented professionals to join our team of experts in data, technology, and finance. If you are interested in a career where you can contribute to the financial infrastructure of North America, we invite you to explore our current opportunities.

### Connect With Us
To learn more about how our analytics services can support your business, or to explore partnership opportunities, visit us at our website or reach out via our contact page.

**Fundata Canada Inc. — Empowering the financial industry with data that matters.**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>